# 05 - Broadcasting

**Broadcasting** is NumPy's way of performing element-wise operations on arrays of **different shapes** — it stretches the smaller array to match the larger one *without copying data*.

### The rules

Shapes are compared **from the right** (trailing dimensions first). Two dimensions are compatible when they are:

1. **equal**, or
2. one of them is **1** (it is stretched), or
3. one of them is **missing** (treated as 1).

```
shape A: (3, 1)        shape A: (3,)
shape B: (1, 4)        shape B: (2, 3)
result : (3, 4)        result : (2, 3)
```

### Why it matters

- `array + 5` (scalar) works by broadcasting.
- Normalising a dataset (`(data - mean) / std`) needs no loops.
- A `(3,1)` column combined with a `(1,4)` row produces every pairwise combination — ideal for grids.

> If shapes can't broadcast, NumPy raises `ValueError: operands could not be broadcast together`. The error prints the shapes for a reason — read it, don't fear it.

In [1]:
import numpy as np

a = np.array([1, 2, 3])

## 1. Scalar + array

The scalar is stretched to every element.

In [2]:
a + 10

array([11, 12, 13])

In [3]:
a * 2

array([2, 4, 6])

## 2. 1D + 1D (same length)

No stretching needed — element-wise.

In [4]:
b = np.array([10, 20, 30])
a + b

array([11, 22, 33])

## 3. Column + row (the classic grid)

A `(3,1)` column and a `(1,4)` row broadcast into a `(3,4)` result — every combination is computed.

In [5]:
col = np.array([[1], [2], [3]])   # shape (3, 1)
row = np.array([[10, 20, 30, 40]])  # shape (1, 4)

print("col shape:", col.shape, " row shape:", row.shape)
col + row

col shape: (3, 1)  row shape: (1, 4)


array([[11, 21, 31, 41],
       [12, 22, 32, 42],
       [13, 23, 33, 43]])

In [6]:
# Reshape a 1D array to a column with the -1 trick
np.arange(3).reshape(-1, 1) + np.arange(4)

array([[0, 1, 2, 3],
       [1, 2, 3, 4],
       [2, 3, 4, 5]])

## 4. Real use: normalising data

Subtract the mean and divide by the standard deviation of each column — a whole dataset in one line.

In [7]:
rng = np.random.default_rng(7)
data = rng.normal(loc=[10, 100, 1000], scale=[1, 10, 100], size=(5, 3))
data

array([[  10.00123015,  102.98745538,  972.58621446],
       [   9.10940816,   95.45329215,  900.8353445 ],
       [  10.0601436 ,  113.40215246,  950.77934814],
       [   9.3795251 ,  104.8984205 , 1035.68870082],
       [  10.10541425,   90.69531955,  997.07481775]])

In [8]:
# Each column now has mean ~0 and std ~1
normalised = (data - data.mean(axis=0)) / data.std(axis=0)
print("means after:", normalised.mean(axis=0).round(3))
print("stds  after:", normalised.std(axis=0).round(3))

means after: [-0.  0. -0.]
stds  after: [1. 1. 1.]


## 5. Broadcast beats loops

Same operation in a Python loop vs broadcast — see how much faster vectorisation is.

In [9]:
import time

big = np.arange(1_000_000)

def loop_version():
    out = np.empty_like(big, dtype=float)
    for i in range(len(big)):
        out[i] = np.sqrt(big[i]) * 2.0
    return out

def broadcast_version():
    return np.sqrt(big) * 2.0

t0 = time.perf_counter(); loop_version(); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); broadcast_version(); t_bc = time.perf_counter() - t0

print(f"loop     : {t_loop:.4f}s")
print(f"broadcast: {t_bc:.6f}s")
print(f"broadcast is {t_loop / t_bc:,.0f}x faster")

loop     : 4.8491s
broadcast: 0.016169s
broadcast is 300x faster


## 6. When shapes do NOT broadcast

Incompatible shapes raise `ValueError` — note the message prints both shapes.

In [10]:
# This fails on purpose: (3,) cannot align with (4,)
try:
    np.arange(3) + np.arange(4)
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (3,) (4,) 


## Summary

- Shapes broadcast when dimensions are **equal** or one is **1** (compared from the right).
- Scalars, `(n,1)` columns, and `(1,m)` rows all stretch automatically.
- Vectorised broadcasting is orders of magnitude faster than loops.
- Incompatible shapes raise `ValueError` with both shapes in the message.

Next: **06-missing-values** — handling `NaN` and infinity.